In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

In [11]:
path = "D:\\Keobuabao"
IMG_SIZE = (224,224)
BATCH_SIZE = 32
SEED = 42

In [12]:
train = keras.utils.image_dataset_from_directory(
    path,
    labels = "inferred",
    label_mode = "int",
    color_mode = "rgb",
    batch_size = BATCH_SIZE,
    image_size = IMG_SIZE,
    seed = SEED,
    validation_split = 0.2,
    subset = "training",
    interpolation = "bilinear",
    pad_to_aspect_ratio = True,
    format = "tf"
)

Found 7324 files belonging to 3 classes.
Using 5860 files for training.


In [13]:
val_test = keras.utils.image_dataset_from_directory(
    path,
    labels = "inferred",
    label_mode = "int",
    color_mode = "rgb",
    batch_size = BATCH_SIZE,
    image_size = IMG_SIZE,
    seed = SEED,
    validation_split = 0.2,
    subset = "validation",
    interpolation = "bilinear",
    pad_to_aspect_ratio = True,
    format = "tf"
)

Found 7324 files belonging to 3 classes.
Using 1464 files for validation.


In [14]:
val_batches = tf.data.experimental.cardinality(val_test)
val = val_test.take(val_batches // 2)
test = val_test.skip(val_batches // 2)

In [15]:
model = keras.Sequential([
    layers.Input(shape = (224,224,3)),

    layers.RandomFlip(mode = "horizon and vertical"),
    layers.RandomRotation(0.3, fill_mode = "reflect"),
    layers.RandomZoom(0.1,0.1, fill_mode = "reflect"),
    layers.RandomBrightness(0.2, value_range = (0,255)),
    layers.RandomContrast(0.2, value_range = (0,255)),

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3,3), padding = "same", activation = "relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), padding = "same", activation = "relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), padding = "same", activation = "relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), padding = "same", activation = "relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(512, (3,3), padding = "same", activation = "relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(256, activation = "relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.5),
    layers.Dense(128, activation = "relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.Dropout(0.4),
    layers.Dense(3, activation = "softmax"),
])

In [16]:
from tensorflow.keras.optimizers import Adam
my_optimizer = Adam(learning_rate=0.0001)
model.compile(
    optimizer = my_optimizer,
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"],
)

model.summary()

early_stop = keras.callbacks.EarlyStopping(monitor = "val_loss", mode = "auto", patience = 3, verbose = 1, restore_best_weights=False)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_1 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_1 (RandomZoom)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness_1             │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 14, 14, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 14, 14, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 7, 7, 512)      │             

 Total params: 8,028,611 (30.63 MB)

 Trainable params: 8,026,627 (30.62 MB)

 Non-trainable params: 1,984 (7.75 KB)

In [17]:
history = model.fit(train, epochs=50, validation_data = val, callbacks=[early_stop])

Epoch 1/50
  1/184 ━━━━━━━━━━━━━━━━━━━━ 11:45 4s/step - accuracy: 0.3125 - loss: 2.3252

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
save_dir = Path('models')
save_dir.mkdir(exist_ok=True) 

model_path = save_dir / 'keobuabao.keras'
model.save(model_path)

NameError: name 'model' is not defined

In [ ]:
import matplotlib.pyplot as plt

# Vẽ biểu đồ Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Độ chính xác của Mô hình')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

# Giả định bạn đã có y_true (nhãn thật) và y_pred (nhãn dự đoán) từ tập Validation/Test
# y_pred = np.argmax(model.predict(val), axis=1) # Cần lấy dữ liệu dự đoán

# (Lưu ý: Để code này chạy mượt, bạn cần loop qua dataset để gom y_true và y_pred, tôi có thể hướng dẫn sâu hơn nếu cần).

# Vẽ ma trận
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Thực tế')
plt.xlabel('Dự đoán')
plt.show()

In [ ]:
import numpy as np
import cv2 
from tensorflow import keras

model = keras.models.loadmodels("C:\\Users\\huynh\\Documents\\ML\\models\\keobuabao1.keras")
class_name = ["Bao", "Bua", "Keo"]

img_path = "C:\\Users\\huynh\\Pictures\\IMG_20260219_181941.jpg"
img = cv2.imread(img_path)

img_resize = cv2.resize(img, (224,224))
img_color = cv2.cvtColor(img_resize, cv2.COLOR_RGB2BGR)

img_input = np.expand_dims(img_color, axis = 0)

predict = model.predict(img_input, verbose = 0)
label = class_name[np.argmax(predict)]
score = np.max(predict)

print(f"ket qua: {label}, ti le: {score}")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# --- CHỈ CẦN THAY ĐỔI ĐƯỜNG DẪN ẢNH TẠI ĐÂY ---
image_to_predict = "C:\\Users\\huynh\\Pictures\\IMG_20260219_181941.jpg"

def run_prediction(img_path, trained_model, dataset):
    # 1. Lấy danh sách tên lớp (Bao, Búa, Kéo) từ dataset
    class_names = train.class_names
    
    # 2. Xử lý ảnh đầu vào
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Biến ảnh thành 4D để khớp với model

    # 3. Dự đoán
    predictions = trained_model.predict(img_array)
    # Lấy index của lớp có xác suất cao nhất
    result_index = np.argmax(predictions[0])
    # Tính toán độ tin cậy (%)
    confidence = 100 * np.max(predictions[0])

    # 4. In kết quả
    print(f"\n[DỰ ĐOÁN]: {class_names[result_index]}")
    print(f"[ĐỘ TIN CẬY]: {confidence:.2f}%")

# Thực hiện lệnh dự đoán
run_prediction(image_to_predict, model, train)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step

[DỰ ĐOÁN]: rock
[ĐỘ TIN CẬY]: 98.64%


In [ ]:
import cv2
import numpy as np
from tensorflow import keras

# 1. Load model duy nhất 1 lần NGOÀI vòng lặp
model = keras.models.load_model('models/keobuabao1.keras')
class_names = ['Bao', 'Bua', 'Keo'] # Đảm bảo đúng thứ tự thư mục lúc train

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 2. TIỀN XỬ LÝ (Giống hệt lúc train)
    # Cắt khung hình vuông ở giữa để tránh bị bóp méo ảnh
    h, w, _ = frame.shape
    start_x = (w - h) // 2
    crop_frame = frame[:, start_x:start_x+h]
    
    img = cv2.resize(crop_frame, (224, 224)) # Câu 1
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Câu 2
    img_input = np.expand_dims(img_rgb, axis=0) # Thêm chiều batch (1, 224, 224, 3)

    # 3. DỰ ĐOÁN
    prediction = model.predict(img_input, verbose=0)
    score = np.max(prediction) # Câu 4
    label = class_names[np.argmax(prediction)]

    # 4. HIỂN THỊ
    color = (0, 255, 0) if score > 0.8 else (0, 0, 255)
    cv2.putText(frame, f"{label}: {score*100:.2f}%", (50, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.rectangle(frame, (start_x, 0), (start_x+h, h), (255, 0, 0), 2)
    
    cv2.imshow('Kéo Búa Bao AI', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): # Câu 3
        break

cap.release()
cv2.destroyAllWindows()

: 